In [ ]:
# Next setting up this new notebook now for running the same model once again but now this time on AWS Sagemaker
# This should follow similar methodology to the part b pattern we'd used in assignment 5
import os
import boto3
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

session = sagemaker.Session()
region = boto3.Session().region_name
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
prefix = 'ana680-bgg-sagemaker-byoc'

print('Region:', region)
print('Role:', role)
print('Bucket:', bucket)

In [ ]:
# Paths are inside this sagemaker folder so this setup is self-contained
local_train_csv = 'sagemaker/data/bgg_master.csv'
if not os.path.exists(local_train_csv):
    raise FileNotFoundError(f'Missing training data at {local_train_csv}')

train_s3_uri = session.upload_data(path=local_train_csv, bucket=bucket, key_prefix=f'{prefix}/train')
print('Train S3 URI:', train_s3_uri)

In [ ]:
# Ensure ECR repo exists
repo_name = 'ana680-bgg-ratings'
ecr = boto3.client('ecr', region_name=region)
try:
    ecr.create_repository(repositoryName=repo_name)
    print('Created ECR repo:', repo_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print('ECR repo already exists:', repo_name)

account_id = boto3.client('sts').get_caller_identity()['Account']
image_uri = f'{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}:latest'
print('Image URI:', image_uri)

In [ ]:
# Build and push container from notebook instance terminal environment
!chmod +x sagemaker/container/bgg_ratings/train sagemaker/container/bgg_ratings/serve
!aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com
!docker build -t {repo_name}:latest sagemaker/container/bgg_ratings
!docker tag {repo_name}:latest {image_uri}
!docker push {image_uri}

In [ ]:
# Train on SageMaker using BYOC estimator
output_path = f's3://{bucket}/{prefix}/output'

estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=output_path,
    sagemaker_session=session,
)

estimator.fit({'train': train_s3_uri})
print('Training complete:', estimator.latest_training_job.name)
print('Model artifacts:', estimator.model_data)

In [ ]:
# Deploy endpoint for quick test
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.t2.medium',
)
predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()
print('Endpoint name:', predictor.endpoint_name)

In [ ]:
# Prediction test using Terra Mystica-like values in feature order
# [mech_roll_spin_and_move, GameWeight, Cat_War, YearPublished]
sample = [[0, 3.9666, 0, 2012]]
pred = predictor.predict(sample)
print('Prediction output:', pred)

In [ ]:
# Cleanup right after testing to avoid charges
predictor.delete_endpoint()
print('Endpoint deleted')

In [ ]:
# Gut-check that the cleanup worked
sm = boto3.client('sagemaker', region_name=region)
for ep in sm.list_endpoints(SortBy='CreationTime', SortOrder='Descending', MaxResults=10).get('Endpoints', []):
    print(ep['EndpointName'], ep['EndpointStatus'])